### Imports

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.neighbors import NearestNeighbors

### Datasets

In [3]:
df_1 = pd.read_csv('datasets/homework_1.1.csv')
df_2 = pd.read_csv('datasets/homework_1.2.csv')

### Problem 1

In [4]:
X = df_1[['X1', 'X2', 'X3']]
Y = df_1['Y']

X = sm.add_constant(X)

model = sm.OLS(Y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        19:53:55   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0026      0.016      0.166      0.8

### Problem 2

In [5]:
multi_model = sm.OLS(Y, sm.add_constant(df_1[['X1', 'X2', 'X3']])).fit()

simple_models = {}
for col in ['X1', 'X2', 'X3']:
    X_simple = sm.add_constant(df_1[col])
    simple_models[col] = sm.OLS(Y, X_simple).fit()

diffs = {}
for col in ['X1', 'X2', 'X3']:
    simple_coef = simple_models[col].params[col]
    multi_coef = multi_model.params[col]
    diff = abs(simple_coef - multi_coef)
    print(f"{col}: {diff:.2f}")

X1: 0.83
X2: 2.12
X3: 0.12


### Problem 4

In [12]:
treated = df_2[df_2['X'] == 1][['Z']].reset_index(drop=True)
control = df_2[df_2['X'] == 0][['Z']].reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['Z']])

distances, indices = nn.kneighbors(treated[['Z']])

print(f"{distances.max():.6f}")

0.210217


### Problem 5

In [7]:
treated_full = df_2[df_2['X'] == 1].reset_index(drop=True)
control_full = df_2[df_2['X'] == 0].reset_index(drop=True)

matched_control_Y = control_full['Y'].iloc[indices.flatten()].values

avg_treated_Y = treated_full['Y'].mean()
avg_matched_control_Y = matched_control_Y.mean()

effect = avg_treated_Y - avg_matched_control_Y

print(f"Effect: {effect:.5f}")

Effect: 0.54336


### Problem 6

In [8]:
nn_radius = NearestNeighbors(radius=0.2)
nn_radius.fit(control_full[['Z']])

distances_r, indices_r = nn_radius.radius_neighbors(treated_full[['Z']])

all_matched_indices = [idx for match in indices_r for idx in match]

matched_series = pd.Series(all_matched_indices)
duplicates = matched_series.duplicated().sum()

print(duplicates)

685


### Problem 7

In [9]:
group_means = []
for match_indices in indices_r:
    if len(match_indices) > 0:
        group_mean = control_full['Y'].iloc[match_indices].mean()
        group_means.append(group_mean)

avg_matched_control_Y = np.mean(group_means)

avg_treated_Y = treated_full['Y'].mean()

effect = avg_treated_Y - avg_matched_control_Y

print(f"Effect: {effect:.4f}")

Effect: 0.5844
